# Processing Data and Converting Excel to Parquet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

Working directory

In [ ]:
os.chdir('..')

In [ ]:
print(os.getcwd())

Import data: static data and sales data

In [ ]:
# Check if the data is already imported
if 'restaurant_data_unprocessed' not in locals():

    # We have two data containers, one for dataframes directly to be added, and another for two parts of dataframes to be merged then added

    # Initialize a dictionary for the 30 Excel files of "restaurant sales data"
    deduplicated_restaurant_data = {}
    partial_data_filenames = []
    for batch_index in [1,2]:

        # Retrieve filnames
        location_filenames = os.listdir(f"deduplicated_orders_item_level")
        location_filenames.remove('CSVs')
        for location_filename in tqdm(location_filenames):
            
            # Keep a version without the file extension
            location_id_with_part = re.sub(r'\.xlsx$', '', location_filename)
            
            # Keep a version without the file extension or the part addition
            location_id = re.sub(r'(_part[12])?\.xlsx$', '', location_filename)

            # To be directly added: check if it is a single file do not add it to be merged
            if "part" not in location_filename:
                df = pd.read_excel(f"deduplicated_orders_item_level/{location_filename}")
                deduplicated_restaurant_data[location_id_with_part] = df

            # To be merged: if it is a partial dataframe, then select the first part
            elif "part1" in location_filename:
                partial_data_filenames.append(location_id)

            # else part 2, then that will be caught on the next loop

    # Read dataframes with a part1 and part2 then merge
    for location_id in tqdm(partial_data_filenames):

            # Add both parts to a list to concat
            to_concat = []
            for part in [1,2]:
                df = pd.read_excel(f"deduplicated_orders_item_level/{location_id}_part{part}.xlsx")
                to_concat.append(df)
            deduplicated_restaurant_data[location_id] = pd.concat(to_concat)



In [ ]:
%store deduplicated_restaurant_data

In [ ]:
totals = []
for loc, df in deduplicated_restaurant_data.items():
    df.duplicated()
    totals.append([loc, df.shape[0]])

print(np.array(totals))

Retrieve unprocessed data

In [ ]:
%store -r static_data_unprocessed
%store -r restaurant_data_unprocessed

Verify the correspondance between location IDs in the reference data and the sales data

In [ ]:
# Static data
locations_batch_1 = static_data_separated['locations_1']['location_id'].value_counts().index.tolist() # Identify the list of location ids for batch 1
locations_batch_2 = static_data_separated['locations_2']['location_id'].value_counts().index.tolist() # Identify the list of location ids for batch 2
locations_static_data = locations_batch_1 + locations_batch_2 # Combine the two batches
locations_static_data.sort()
locations_df_static_data = pd.DataFrame(locations_static_data, columns=['location_id'])

# Sales data
location_filenames = os.listdir("palate_data_excel/batch_1/orders_item_level") + os.listdir("palate_data_excel/batch_2/orders_item_level") # Concatenate the filenames
locations_sales_data = [re.sub(r'\.xlsx$', '', filename) for filename in location_filenames] # Remove file endings
locations_df_sales_data = pd.DataFrame(locations_sales_data, columns=['location_id'])

# Match
location_id_matching = pd.concat([locations_df_static_data, locations_df_sales_data], axis=1)

View data columns

In [ ]:
# Initialize a list for rows of column names
data_columns = []
for name, df in static_data_separated.items():

    # Add row of names to check that column matches
    data_columns.append([name] + df.columns.tolist())

# Same for sales data
for name, df in restaurant_data.items():

    # Add row of names to check that column matches
    data_columns.append([name] + df.columns.tolist())

# Make dataframe for viewing
data_columns_df = pd.DataFrame(data_columns)

Separate data parts 1 and 2 to merge

In [ ]:
# Separate out items_tagged, so that v1 and v2 can be crosschecked
items_tagged_1 = static_data_separated['items_tagged_1']
items_tagged_v2_1 = static_data_separated['items_tagged_v2_1']
items_tagged_2 = static_data_separated['items_tagged_2']

# Create a new dict
static_data_without_items_tagged = static_data_separated.copy()

# Remove the items_tagged data from the new dict
if 'items_tagged_1' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_1']
if 'items_tagged_v2_1' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_v2_1']
if 'items_tagged_2' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_2']

# Initialize lists for the two parts
static_data_batch_1 = []
static_data_batch_2 = []
for name, df in static_data_without_items_tagged.items():

    # Add batch 1 numbering
    if "1" in name:
        df.columns.name = name
        df['batch'] = 1
        static_data_batch_1.append(df)

    # Add batch 2 numbering
    else:
        df.columns.name = name
        df['batch'] = 2
        static_data_batch_2.append(df)

Compare batch 1's items_tagged v1 shape versus v2 shape

In [ ]:
# Tagged items data batch 1 v1 
print(items_tagged_1.shape)

# Tagged items data batch 1 v2
print(items_tagged_v2_1.shape)

# Tagged items data batch 2
print(items_tagged_2.shape)

Compare batch 1's items_tagged v1 values versus v2 values

In [ ]:
# Prepare to standardize formatting
items_tagged_1_modified = items_tagged_1.copy()
items_tagged_v2_1_modified = items_tagged_v2_1.copy()

# Match the part of version 2 that corresponds with verison 1
items_tagged_v2_1_modified = items_tagged_v2_1_modified.iloc[:items_tagged_1.shape[0], :]

# Standardize formatting
items_tagged_1_modified['is_plant_based'] = items_tagged_1_modified['is_plant_based'].str.lower().str.replace('.', '')
items_tagged_v2_1_modified['is_plant_based'] = items_tagged_v2_1_modified['is_plant_based'].str.lower().str.replace('.', '')

# NaNs are always treated as unequal, so temporarily fill them
items_tagged_1_modified.fillna(0, inplace=True)
items_tagged_v2_1_modified.fillna(0, inplace=True)

# Check if there are differences
changes = (items_tagged_1_modified != items_tagged_v2_1_modified).apply(lambda r: r.any(), axis=1)
changes.sum()

# There are no differences so v1 has strictly less data and is useless, and we won't carry it forward in the data processing

Combine data parts 1 and 2

In [ ]:
# Reinitialize dict for static data
static_data = {}

# Standardize number of columns: version two has an item description while version 1 does not
items_tagged_v2_1['item_description'] = np.nan
items_tagged_v2_1 = items_tagged_v2_1[items_tagged_2.columns.tolist()]

# Merge
items_tagged = pd.concat([items_tagged_v2_1, items_tagged_2])
items_tagged.reset_index(drop=True, inplace=True)

# First add the items_tagged menu data
static_data['items_tagged'] = items_tagged.copy()

# Merge the rest and add
for df1, df2 in zip(static_data_batch_1, static_data_batch_2):

    # Check number of rows in the two parts
    print(df1.columns.name, df1.shape)
    print(df2.columns.name, df2.shape)

    # Concat
    df = pd.concat([df1, df2])
    df.reset_index(drop=True, inplace=True)

    # Remove the part for the final name
    df.columns.name = re.sub(r'_[12]', '', df1.columns.name)
    
    static_data[df.columns.name] = df.copy()

Format minimal typing in columns to export to parquet

In [ ]:
# Standardize labeling formatting
static_data['items_tagged']['is_plant_based'] = static_data['items_tagged']['is_plant_based'].str.lower().str.replace('.', '')

# Label values won't be modified, so make category
static_data['items_tagged']['is_plant_based'] = static_data['items_tagged']['is_plant_based'].astype('category')

# Fix encoding issues for parquet, keeping NaNs
static_data['items_tagged']['item_name'] = static_data['items_tagged']['item_name'].apply(lambda x: str(x) if not pd.isna(x) else x)
static_data['locations']['zip_code'] = static_data['locations']['zip_code'].apply(lambda x: str(x) if not pd.isna(x) else x)

# Convert to datetime
static_data['before_after_details']['cross_over_date'] = pd.to_datetime(static_data['before_after_details']['cross_over_date'])

# Set indices
static_data['locations'].set_index(keys='location_id', inplace=True)
static_data['before_after_details'].set_index(keys='location_id', inplace=True)

# Sales data
for loc_id, df in tqdm(restaurant_data.items()):

    # Fix encoding issues for parquet, keeping NaNs
    df['item_modifications'] = df['item_modifications'].apply(lambda x: str(x) if not pd.isna(x) else x)
    df['item_name'] = df['item_name'].apply(lambda x: str(x) if not pd.isna(x) else x)

    # ID values won't be modified, so make category
    df['unique_id'] = df['unique_id'].astype('category')
    df['order_id'] = df['order_id'].astype('category')
    df['location_id'] = df['location_id'].astype('category')
    df['customer_id'] = df['customer_id'].astype('category')

    # Make time series and remove the alphabetical timezone identifier (there is a numerical one already)
    df['created_at'] = df['created_at'].str.replace(r'\s[A-Z]{3}', '', regex=True)
    df['created_at'] = pd.to_datetime(df['created_at'], utc=True)
    df.set_index(keys="created_at", drop=True, inplace=True)
    df.sort_index(inplace=True)

    # Add a unit price column
    df['unit_price'] = df['item_price'] / df['item_quantity']

    # Save
    restaurant_data[loc_id] = df

Export to parquet

In [ ]:
# Note: (most commpressed) gzip > zstd > snappy > lz4 (fastest compression)

# Static data
for name, df in tqdm(static_data.items()):

    # Write to parquet
    df.to_parquet(f"palate_data_parquet/{name}.parquet", compression='zstd', index=True)

# Sales data
for loc_id, df in tqdm(restaurant_data.items()):

    # Write to parquet
    df.to_parquet(f"palate_data_parquet/orders_item_level/{loc_id}.parquet", compression='zstd', index=True)

Data count

In [ ]:
total = 0
totals = []
for loc, df in restaurant_data.items():
    df.duplicated()
    totals.append(df.shape[0])
    total += df.shape[0]
print(total)
print(np.array([totals]).T)